# Stage 2 nvBench post-query preparation

Mount Drive, install the repository package, prepare nvBench as materialized post-query examples, and verify output artifacts.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path(os.environ.get('T2V_DRIVE_ROOT', '/content/drive/MyDrive/diploma/petr_text_to_visualization_part'))
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

repo_candidates = [
    Path(os.environ['T2V_REPO_ROOT']) if os.environ.get('T2V_REPO_ROOT') else None,
    DRIVE_ROOT / 'repo',
    DRIVE_ROOT / 'NL2BI-AI-assistant',
    Path('/content/NL2BI-AI-assistant'),
    Path('/content/petukhov_t2v_repo'),
]
repo_candidates = [path for path in repo_candidates if path is not None]
repo_root = next((path for path in repo_candidates if (path / 'pyproject.toml').exists()), None)
if repo_root is None:
    print('STAGE2_SETUP_BLOCKED')
    print(json.dumps({
        'reason': 'Colab-visible repository checkout was not found; pyproject.toml root marker is missing',
        'checked': [str(path) for path in repo_candidates],
        'hint': 'Copy or clone the current repository into Drive or /content, then set T2V_REPO_ROOT to that checkout before running benchmark preparation.',
    }, ensure_ascii=False, indent=2))
    raise RuntimeError('T2V_REPO_ROOT is required for Stage 2 Colab run')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_root)])
os.environ['T2V_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['T2V_REPO_ROOT'] = str(repo_root)
print('STAGE2_SETUP_OK')
print(json.dumps({'drive_root': str(DRIVE_ROOT), 'repo_root': str(repo_root)}, ensure_ascii=False, indent=2))


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'prepare_nvbench.py'),
    '--drive-root', str(drive_root),
    '--sample-size', '20',
    '--min-successful', '1',
    '--json',
]
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'sample20 preparation failed with exit code {result.returncode}')
print('STAGE2_SAMPLE20_OK')


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'prepare_nvbench.py'),
    '--drive-root', str(drive_root),
    '--sample-size', '200',
    '--min-successful', '50',
    '--json',
]
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'sample200 preparation failed with exit code {result.returncode}')
print('STAGE2_SAMPLE200_OK')


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
processed = drive_root / 'datasets' / 'processed' / 'nvbench_postquery'
required = [
    processed / 'examples.jsonl',
    processed / 'examples_sample200.jsonl',
    processed / 'dataset_card.md',
    processed / 'prepare_result.json',
    processed / 'tables',
]
missing = [str(path) for path in required if not path.exists()]
summary = json.loads((processed / 'prepare_result.json').read_text(encoding='utf-8')) if (processed / 'prepare_result.json').exists() else {}
table_count = len(list((processed / 'tables').glob('*.csv'))) if (processed / 'tables').exists() else 0
print(json.dumps({'processed': str(processed), 'missing': missing, 'summary': summary, 'table_count': table_count}, ensure_ascii=False, indent=2))
if missing:
    raise RuntimeError(f'Missing Stage 2 artifacts: {missing}')
if summary.get('successful_examples', 0) < 50:
    raise RuntimeError('Less than 50 successful examples after sample200 preparation')
print('STAGE2_ARTIFACTS_OK')
